## 安裝套件

In [1]:
# 錯誤率計算工具
!pip install jiwer

# Hugging Face資料集函式庫
!pip install datasets

# 斷詞器
!pip install jiaba

!pip install requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 10.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 15.8 MB/s eta 0:00:00


## 快速測試 jieba

In [2]:
import jieba

# Simple test function
def debug_chinese_tokenization(text):
    # Remove punctuation
    punct = "，。！？；：""''（）【】《》、…"
    for p in punct:
        text = text.replace(p, '')

    text = text.replace('\n', '')
    # Print text after punctuation removal
    print("After punctuation removal:", repr(text))

    # Tokenize and print words
    words = list(jieba.cut(text))
    print("After jieba tokenization:", words)

    # Create space-separated string
    result = ' '.join(words)
    print("Final result:", repr(result))
    return result

# Test with a small sample
sample_text = "今天天气很好。\n我想去公园走走。"
debug_chinese_tokenization(sample_text)

Building prefix dict from the default dictionary ...
DEBUG:jieba:Building prefix dict from the default dictionary ...


After punctuation removal: '今天天气很好我想去公园走走'


Dumping model to file cache /tmp/jieba.cache
DEBUG:jieba:Dumping model to file cache /tmp/jieba.cache
Loading model cost 1.649 seconds.
DEBUG:jieba:Loading model cost 1.649 seconds.
Prefix dict has been built successfully.
DEBUG:jieba:Prefix dict has been built successfully.


After jieba tokenization: ['今天天气', '很', '好', '我', '想', '去', '公园', '走走']
Final result: '今天天气 很 好 我 想 去 公园 走走'


'今天天气 很 好 我 想 去 公园 走走'

In [3]:
from typing import List,Dict

## 請換成自己的待測檔

In [5]:
# 自行上傳檔案
# refpath: 原稿
# hypopath: 逐字稿
def read_files():
    refpath = 'sb_reference.txt'
    hypopath = 'sb_hypothesis.txt'
    nfstrs=[]
    for fpath in [refpath, hypopath]:
        with open(fpath, 'r', encoding='utf-8') as f:
            fstr = f.read()
        nfstrs.append(fstr)
    return nfstrs


# 檔案來自網路連結
# reflink: 原稿
# hypolink: 逐字稿

def read_links():
    import requests
    hypolink='https://gist.github.com/timwu-ipevo/6a7fb0b6547e0d83b05a13e7d703e8ba/raw/f92a8efc257ba339af50bd68a1efad7449120a51/sb_hypothesis.txt'
    reflink='https://gist.github.com/timwu-ipevo/6a7fb0b6547e0d83b05a13e7d703e8ba/raw/f92a8efc257ba339af50bd68a1efad7449120a51/sb_reference.txt'

    refstr = requests.get(reflink).text
    hypostr = requests.get(hypolink).text
    return [refstr, hypostr]

In [9]:
# rewrite ChineseTransform as a function
def chinese_transform(texts:List[str])->List[str]:
    #chinese_punc = "，。！？；：""''（）【】《》、…"
    chinese_punc = "，。！？；：" "''（）【】《》、…呢啊呀欸吶"

    newtexts = []
    for text in texts:
        text = text.strip()
        # Remove Chinese and English punctuation
        for punct in chinese_punc:
            text = text.replace(punct, '')
        # Segment Chinese text into words using jieba
        words = jieba.cut(text, cut_all=False, HMM=False)
        # Join with spaces to make it compatible with jiwer
        newtext = ' '.join(words)
        newtexts.append(newtext)
    return newtexts



In [10]:
import jiwer
def to_multi_lines(texts):
    return [text for text in texts.split("\n") if text.strip() != ""]

def calculate_chinese_wer2( ground_truth:str, hypothesis:str, hslice):
    if True:
        ground_truth_mlines =  to_multi_lines(ground_truth)
        hypothesis_mlines = to_multi_lines(hypothesis)
        print('ground truth:')
        ground_truth_mlines = chinese_transform(ground_truth_mlines[hslice])
        print('hypothesis:')
        hypothesis_mlines = chinese_transform(hypothesis_mlines[hslice])

        ground_truth = ' '.join(ground_truth_mlines)
        hypothesis = ' '.join(hypothesis_mlines)


    out = jiwer.process_words( ground_truth, hypothesis )
    print(jiwer.visualize_alignment(out))

[refstr, hypostr] = read_files()


# 開始正式測試

In [11]:
# 測前兩句
calculate_chinese_wer2( refstr, hypostr, slice(0,2))


ground truth:
hypothesis:
sentence 1
REF: * 大家 吉祥 今天 要 跟 大家 談 人生 行路
HYP: 敗 大家 吉祥 今天 要 跟 大家 談 人生 行路
     I                        

number of sentences: 1
substitutions=0 deletions=0 insertions=1 hits=9

mer=10.00%
wil=10.00%
wip=90.00%
wer=11.11%



In [12]:
# 測前五句
calculate_chinese_wer2( refstr, hypostr, slice(0,5))


ground truth:
hypothesis:
sentence 1
REF: * 大家 吉祥 今天 要 跟 大家 談 人生 行路 這 是 出自 * 老舍 的 一 個 文章 老舍 本身 是 一位 文 學 家 他 一生 可以 說 留下 很多
HYP: 敗 大家 吉祥 今天 要 跟 大家 談 人生 行路 這 是 出自 老  神 的 一 個 文章 老舍 本身 是 一位 文 學 家 * ** ** * ** **
     I                                I  S                           D  D  D D  D  D

number of sentences: 1
substitutions=1 deletions=6 insertions=2 hits=23

mer=28.12%
wil=32.18%
wip=67.82%
wer=30.00%



In [13]:
# 測前十句
calculate_chinese_wer2( refstr, hypostr, slice(0,10))


ground truth:
hypothesis:
sentence 1
REF: * 大家 吉祥 今天 要 跟 大家 談 人生 行路 這 是 出自 * 老舍 的 一 個 文章 老舍 本身 是 一位 文 學 家 他 一生 可以 說 留下 很多 好 的 小 說 好 的 文章 讓 大家 都 非常 非常 的 受用 歡 喜 * 這 篇文章 裡 面 有 講 到 才 華 是 刀刃 辛苦 是 ** 磨刀石 也 就是 我 們 也 常 講
HYP: 敗 大家 吉祥 今天 要 跟 大家 談 人生 行路 這 是 出自 老  神 的 一 個 文章 老舍 本身 是 一位 文 學 家 他 一生 可以 說 留下 很多 好 的 小 說 好 的 文章 讓 大家 都 非常 非常 的 受用 歡 喜 那 這 篇文章 裡 面 有 講 到 才 華 是 刀刃 辛苦 是 磨刀   使 也 就是 * * * * *
     I                                I  S                                                                                I                                I   S      D D D D D

number of sentences: 1
substitutions=2 deletions=5 insertions=4 hits=60

mer=15.49%
wil=18.59%
wip=81.41%
wer=16.42%



In [14]:
# 測全文
calculate_chinese_wer2( refstr, hypostr, slice(0,300))

ground truth:
hypothesis:
sentence 1
REF: * 大家 吉祥 今天 要 跟 大家 談 人生 行路 這 是 出自 * 老舍 的 一 個 文章 老舍 本身 是 一位 文 學 家 他 一生 可以 說 留下 很多 好 的 小 說 好 的 文章 讓 大家 都 非常 非常 的 受用 歡 喜 * 這 篇文章 裡 面 有 講 到 才 華 是 刀刃 辛苦 是 ** 磨刀石 也 就是 我 們 也 常 講 所 謂 天才 是 * 一分 的 天分 九十九 分 的 努力 再 鋒 利 的 刀刃 若 日 久 不 磨 也 會 生 銹 也 表示 我 們 在 日常生活 裡 面 不管 是 面 對  工作 面 對 事 業 乃至 人 我 之 間 都 要 非常 的 努力 都 要 很 專 心 的 去 經 營 才 不 會 從 中 能 夠 有所 差 錯 所以 在 這 當 中 也 看出 ** 凡事 必 須 要 勤 勞 凡事 也 必 須 要 專 心 我 們 講 一生 之 計 * 在 於 勤 這 一 輩 子 不管 再 怎 麼 樣 總 是 要 勤 勞 我 也 經 常 講 一般 人 一般 人 以 為 年 輕 就是 本 錢 但 事 實 上 如果 說 一 個 年 輕 人 他 不 懂得 努力 不 懂得 勤 勞 我 想 那 也 非常 非常 可惜 所以 真正 的 本 錢 應 該 * 就是 我 們 一 顆 勤 勞 的 心 能 夠 面 對 種 種 的 事 面 對 種 種 的 人 也 就是 好好 的 去 經 營 好好 的 去 面 對 每 一 個 小 事情 你 都 很 用心 每 一 個 小 事情 你 都 不要 去 錯 過 我 想 這 樣 生活 確 實 能 夠 踏 實 * * 在 生活 裡 面 好 的 念 頭 也 非常 非常 重要 一 個 念 頭 我 們 或 許 會 覺 得 沒 什 麼 但是 好 的 念 頭 的 聚集 它 確 實 會 發 生 很大 的 力量 所以 我 們 常 講 細 水 長 流 穿破 * 石 熱 湯 停火 易 成 冰 這 也 表示 我 們 在 生活 裡 面 確 實 要 非常 非常 的 勤 勞 所 謂 滾 動 之 石 不易 生 苔 這 也 就是 我 們 生活 應 該 要 有 勤 勞 的 這 份 精神 謙 虛 使 人 的 心 縮 小 像 一 個 小石 卵 雖 然 小 而 極 結 實 結 實 才能 

# 【語音辨識 - Whisper】 準確與否需要有一把 📏尺來衡量


前面我們介紹了幾個關於Whisper的基本概念，這裡附上 [🚀傳送門](https://vocus.cc/article/644526c8fd89780001ffdd9f) ，歡迎好好閱讀一番，但我們除了學會如何用語音辨識的工具之外，「準確率」對我們來說也是一個非常重要的一環，但我們究竟應該要如何評估所謂的準確率呢？ 不知道沒關係，當您看完這個篇章就能夠學會如何計算文字的「字元錯誤率」、「字詞錯誤率」...，非常值得您細細品嘗與學習，就讓我們往下一步步的完成評估準確率的程序吧！

這次的評估工具我們會使用jiwer這一套來進行說明，它支援了多種的計算方式，包括： WER、CER、MER...等，那這些計算方式各有什麼不同呢？ 就讓我們繼續看下去吧！

#### 詞錯誤率 Word Error Rate(WER)
WER是以「詞」為單位進行計算，它用來衡量句子中有多少詞彙需要進行修改才能和正確答案一樣。

```bash
公式: (S + D + I) / (H + S + D)
計算過程: (2 + 0 + 1) / (2 + 2 + 0)
3 / 4 ≈ 75%。
```

💡 既然是以`詞`為單位的話，那麼我們的答案與辨識結果請先進行斷詞(通常用空白隔開)， 標點符號也是考量的因素之一喔。

#### 平均錯誤率 Mean Error Rate(MER)
這項指標與WER主要差別在於分母的部分尚未將`Insertion`給考量進來計算，因為它衡量的不僅是詞彙層級，而是句子層級，因此會更加全面。

```bash
公式： (S + D + I) / (H + S + D + I)
計算過程： (2 + 0 + 1) / (2 + 2 + 0 + 1)

3 / 5 ≈ 60%
```

#### 詞保留率 Word Information Preservation(WIP)
這項指標主要在評估我們的辨識結果究竟有多少比例的字詞是一模一樣完全正確的。

```bash
num_rf_words = 正確答案字詞數 = 4
num_hp_words = 辨識結果字詞數 = 5
公式： (H / num_rf_words) * (H / num_hp_words)
計算過程: (2 / 4) * (2 / 5)
0.5 * 0.4 ≈ 20%
```
#### 詞漏失率 Word Information Lost(WIL)
既然有詞的保留率，那麼相反的就是漏失率，因此上述的結果得出之後，用1減去保留率就是漏失率，可以粗略的評估總共漏失了多少比率。
```bash
公式: 1 - wip
1 - 0.2 ≈ 80%
```